# SCBE Code-Diffusion Bake-Off (Colab T4)

Self-contained run of the v6h coding gate against THREE generators:
1. **AR baseline**: `Qwen/Qwen2.5-Coder-7B-Instruct` (greedy decode)
2. **Schrödinger oracle**: same Qwen base + active-subset wave evolution under contract potential V = -log P_base - alpha*required + beta*forbidden
3. **Diffusion candidate (optional)**: `apple/DiffuCoder-7B-Instruct`

All three run on T4 (16 GB) — load → run 12 prompts → unload → next.
Same `_gate_score` rule as production v6h. Triangulates by problem shape (8 buckets).

**Run all cells.** Results push to private HF dataset `issdandavis/scbe-diffusion-bakeoff-results`.

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch transformers accelerate huggingface_hub numpy
print('deps installed')

In [ ]:
# Cell 2: GPU check
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}, VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('No GPU! Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3: HuggingFace auth
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('using Colab secret HF_TOKEN')
except Exception:
    HF_TOKEN = input('paste HF token: ').strip()
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN)
print('logged in')

In [ ]:
# Cell 4: 12-prompt contract (inlined from config/eval/coding_diffusion_bakeoff_v1.json)
CONTRACT = {
  'contract_id': 'coding_diffusion_bakeoff_v1',
  'source_contract': 'coding_verification_unseen_eval_v2',
  'shape_taxonomy': {
    'simple_implementation': 'Single-tongue body, no cross-translation.',
    'body_fidelity_with_guards': 'Single tongue with explicit guard branches.',
    'cross_tongue_translation': 'Source given in tongue X; target in tongue Y with slot map.',
    'multi_lens_parallel': 'Same algorithm in MULTIPLE tongues in one response.',
    'meta_identification': 'Identify-and-describe rather than implement.',
    'structured_card': 'Fixed-shape verdict bundle: enum verdict + horizon + evidence.',
    'route_decision_with_body': 'Tongue routing decision + canonical implementation + reason.',
    'negative_constraint': 'Forbidden vocabulary dominates contract.',
  },
  'prompts': [
    {'id': 'code_eval_inventory_unique_python', 'shape': 'simple_implementation',
     'prompt': "Implement inventory_unique(items) in tongue KO (Kor'aelin/Python). It must return a list of unique items in first-seen order. Use a seen-set guard. Show the function signature, the seen-set initialization, the loop, the membership check, and the return.",
     'required': ['def inventory_unique', 'items', 'seen', 'for ', 'if ', 'not in', 'append', 'return'],
     'forbidden': ['TODO', 'planned', 'RDKit', 'SMILES', 'valence']},
    {'id': 'code_eval_count_vowels_translate', 'shape': 'cross_tongue_translation',
     'prompt': "Algorithm: count_vowels (count the number of vowels in a string). Source tongue: KO (Kor'aelin, Python):\n\n```py\ndef count_vowels(s):\n    total = 0\n    for ch in s:\n        if ch in 'aeiouAEIOU':\n            total += 1\n    return total\n```\n\nTranslate to tongue UM (Umbroth, Haskell), preserving slot alignment (sig, init, loop_open, loop_body, ret). Mark each slot in the output.",
     'required': ['count_vowels', 'umbroth', 'haskell', 'sig', 'init', 'loop_open', 'loop_body', 'ret'],
     'forbidden': ['def count_vowels', 'function countVowels', 'fn count_vowels', 'TODO', 'planned']},
    {'id': 'code_eval_zero_guard_safe_subtract', 'shape': 'body_fidelity_with_guards',
     'prompt': "Implement safe_subtract(a, b) in tongue KO (Kor'aelin/Python). It must return None when either argument is None, otherwise return a - b. Show the explicit None guard and the subtraction.",
     'required': ['def safe_subtract', 'a', 'b', 'if ', 'none', 'return', 'a - b'],
     'forbidden': ['TODO', 'planned', 'raise', 'valence', 'molecule']},
    {'id': 'code_eval_clamp_value_rust', 'shape': 'body_fidelity_with_guards',
     'prompt': 'Implement clamp_value(x, lo, hi) in tongue RU (Runethic/Rust) returning x clamped into the inclusive range [lo, hi]. Show the function signature with i64 types, the lower-bound branch, the upper-bound branch, and the otherwise return.',
     'required': ['fn clamp_value', 'i64', 'if ', 'x < lo', 'x > hi', 'return'],
     'forbidden': ['def clamp_value', 'function clampValue', 'TODO', 'planned']},
    {'id': 'code_eval_avali_javascript_lens', 'shape': 'simple_implementation',
     'prompt': "Implement first_word(s) in tongue AV (Avali/JavaScript). It must return the first whitespace-delimited token of the input string, or an empty string when the input is empty. Use export function syntax. Show the empty-input guard, the split, and the return.",
     'required': ['export function firstWord', 'if ', 'split', 'return', "''"],
     'forbidden': ['def first_word', 'fn first_word', 'TODO', 'planned']},
    {'id': 'code_eval_identify_algorithm_haskell', 'shape': 'meta_identification',
     'prompt': 'Identify the algorithm and its slot structure from this snippet (UM, Haskell):\n\n```hs\ndoubleAll :: [Int] -> [Int]\ndoubleAll xs = map (*2) xs\n```\n\nReturn algorithm name, description, tongue with phi-weight, and the slot list.',
     'required': ['algorithm:', 'double', 'umbroth', 'phi=6.85', 'slots:', 'sig', 'body'],
     'forbidden': ["kor'aelin", 'runethic', 'TODO', 'valence']},
    {'id': 'code_eval_multi_lens_consistency', 'shape': 'multi_lens_parallel',
     'prompt': "Implement triple(x) so it returns 3 * x. Provide three language lenses: KO (Kor'aelin/Python), AV (Avali/JavaScript), RU (Runethic/Rust). Each lens must show the function signature, the multiplication body, and the return. Mark each lens with its tongue label.",
     'required': ["kor'aelin", 'avali', 'runethic', 'def triple', 'function triple', 'fn triple', '* 3', 'return'],
     'forbidden': ['TODO', 'planned', 'valence', 'molecule']},
    {'id': 'code_eval_approval_card_verdict', 'shape': 'structured_card',
     'prompt': "Evaluate this agentic task-flow card before execution.\n\ntitle: Inventory Audit Runner\nscript_path: scripts/system/inventory_audit_runner.py\ncommand: python scripts/system/inventory_audit_runner.py\npurpose: Walks a stockroom CSV, flags duplicate SKUs, exports a corrected CSV.\ncard_route: DR / Draumric Markdown\nscript_route: KO / Kor'aelin / Python\nroute_reason: file_io\n\nReturn an explicit verdict (PROMOTE/HOLD/INCUBATE/TRANSFORM/ESCALATE/DENY/ARCHIVE), evidence requirement, next safe action, and whether this is fast, medium, or long return.",
     'required': ['verdict', 'evidence', 'next', 'horizon', 'draumric', "kor'aelin"],
     'forbidden': ['TODO', 'valence', 'RDKit', 'SMILES']},
    {'id': 'code_eval_geoseal_pair_route', 'shape': 'route_decision_with_body',
     'prompt': "A user asks the SCBE coding agent: 'Write a Python function `running_average(values)` that returns a list of the running mean of the input numeric list.' Provide the route decision (which tongue lens), the canonical Python implementation including the running-sum accumulator and divisor by index, and a one-line note on why KO is the appropriate routing tongue for this task.",
     'required': ["kor'aelin", 'def running_average', 'values', 'total', 'for ', 'append', 'return', '/'],
     'forbidden': ['TODO', 'planned', 'RDKit', 'SMILES', 'valence']},
    {'id': 'code_eval_lane_boundary_no_chem', 'shape': 'negative_constraint',
     'prompt': "A user pastes the code token `queue_drain_guard` into the SCBE coding agent and asks: 'Verify this token.' Reply ONLY in code-side terms. Do NOT mention chemistry vocabulary at all (including in negation). Your response must (a) classify queue_drain_guard as a code identifier, (b) state the next action is to grep or search for its definition in the source tree, and (c) state that the unit test exercising it must be run. Treat the input as a symbol-resolution task, not a chemistry task.",
     'required': ['queue_drain_guard', 'code identifier', 'definition', 'unit test', 'run'],
     'forbidden': ['valence', 'RDKit', 'SMILES', 'molecule', 'atom', 'chemistry', 'pentavalent', 'ester']},
    {'id': 'code_eval_executable_dict_merge', 'shape': 'body_fidelity_with_guards',
     'prompt': "Implement merge_counts(a, b) in tongue KO (Kor'aelin/Python). Both arguments are dicts mapping str to int. Return a new dict whose keys are the union, and each value is the sum of the values from a and b (treat missing as 0). Show the new-dict initialization, the iteration over both inputs, the get-with-default, and the return.",
     'required': ['def merge_counts', 'a', 'b', 'result', 'for ', 'get(', ', 0)', 'return'],
     'forbidden': ['TODO', 'planned', 'RDKit', 'valence']},
    {'id': 'code_eval_runethic_option_chain', 'shape': 'body_fidelity_with_guards',
     'prompt': 'Implement first_positive(xs) in tongue RU (Runethic/Rust) returning Option<i64>. It returns Some of the first positive integer in the slice, or None if no positive integer exists. Show the function signature with Option<i64> return type, the iter().find pattern OR an explicit for-loop with early return, and the None fallback.',
     'required': ['fn first_positive', 'i64', 'Option', 'Some', 'None', '> 0'],
     'forbidden': ['def first_positive', 'function firstPositive', 'TODO', 'planned']},
  ]
}
print(f'{len(CONTRACT["prompts"])} prompts loaded')

In [ ]:
# Cell 5: Canonical v6h gate scorer
import re

def gate_score(prompt, response):
    body_lower = (response or '').lower()
    missing_required = [str(t) for t in (prompt.get('required') or []) if str(t).lower() not in body_lower]
    def contains_forbidden(term):
        needle = str(term).strip().lower()
        if not needle:
            return False
        if re.fullmatch(r'[a-z0-9_ -]+', needle):
            body = r'\s+'.join(re.escape(p) for p in needle.split())
            pat = r'(?<![a-z0-9_])' + body + r'(?![a-z0-9_])'
            return re.search(pat, body_lower) is not None
        return needle in body_lower
    triggered_forbidden = [str(t) for t in (prompt.get('forbidden') or []) if contains_forbidden(t)]
    return {
        'id': prompt.get('id'),
        'shape': prompt.get('shape', 'unknown'),
        'ok': (not missing_required) and (not triggered_forbidden),
        'missing_required': missing_required,
        'triggered_forbidden': triggered_forbidden,
        'response_chars': len(response or ''),
    }
print('scorer loaded')

In [ ]:
# Cell 6: Schrödinger primitive — active-subset wave evolution
import numpy as np
from dataclasses import dataclass

@dataclass
class SchrodingerConfig:
    alpha_required: float = 1.5
    beta_forbidden: float = 4.0
    inverse_mass: float = 0.3
    tau: float = 0.25
    n_steps: int = 8
    sample: bool = False
    imaginary_time: bool = True
    active_top_k: int = 256

def _build_indicator_masks(vocab, required, forbidden):
    req = [str(r).lower().strip() for r in (required or []) if str(r).strip()]
    forb = [str(f).lower().strip() for f in (forbidden or []) if str(f).strip()]
    n = len(vocab)
    req_mask = np.zeros(n, dtype=np.float32)
    forb_mask = np.zeros(n, dtype=np.float32)
    forb_patterns = []
    for f in forb:
        if re.fullmatch(r'[a-z0-9_ -]+', f):
            body = r'\s+'.join(re.escape(p) for p in f.split())
            forb_patterns.append(re.compile(r'(?<![a-z0-9_])' + body + r'(?![a-z0-9_])'))
        else:
            forb_patterns.append(re.compile(re.escape(f)))
    for i, raw in enumerate(vocab):
        tok = (raw or '').lower().strip()
        if not tok:
            continue
        for r in req:
            if r and (r in tok or tok in r):
                req_mask[i] += 1.0
                break
        for pat in forb_patterns:
            if pat.search(tok):
                forb_mask[i] += 1.0
                break
    return req_mask, forb_mask

def _select_active_subset(base_logits, req_mask, top_k):
    n = base_logits.shape[0]
    if top_k <= 0 or top_k >= n:
        return np.arange(n, dtype=np.int64)
    k = min(int(top_k), n)
    top_idx = np.argpartition(-base_logits, k - 1)[:k]
    req_idx = np.nonzero(req_mask > 0)[0]
    return np.unique(np.concatenate([top_idx, req_idx])).astype(np.int64)

def _evolve(psi, V, cfg, k_grid):
    factor = 1.0 if cfg.imaginary_time else 1j
    half_V = np.exp(-factor * cfg.tau * V / 2.0)
    kin = np.exp(-factor * cfg.tau * (k_grid ** 2) * cfg.inverse_mass / 2.0)
    for _ in range(int(cfg.n_steps)):
        psi = half_V * psi
        psi = np.fft.ifft(kin * np.fft.fft(psi))
        psi = half_V * psi
        if cfg.imaginary_time:
            norm = np.linalg.norm(psi)
            if norm > 0:
                psi = psi / norm
    return psi

class SchrodingerLogitsProcessor:
    def __init__(self, req_mask, forb_mask, cfg):
        self.req_mask = req_mask
        self.forb_mask = forb_mask
        self.cfg = cfg
    def __call__(self, input_ids, scores):
        np_scores = scores[0].detach().to('cpu', dtype=torch.float32).numpy()
        if np_scores.shape[0] != self.req_mask.shape[0]:
            if np_scores.shape[0] > self.req_mask.shape[0]:
                pad = np_scores.shape[0] - self.req_mask.shape[0]
                self.req_mask = np.concatenate([self.req_mask, np.zeros(pad, dtype=self.req_mask.dtype)])
                self.forb_mask = np.concatenate([self.forb_mask, np.zeros(pad, dtype=self.forb_mask.dtype)])
            else:
                self.req_mask = self.req_mask[:np_scores.shape[0]]
                self.forb_mask = self.forb_mask[:np_scores.shape[0]]
        active = _select_active_subset(np_scores, self.req_mask, self.cfg.active_top_k)
        sub_logits = np_scores[active]
        sub_req = self.req_mask[active].astype(np.float64)
        sub_forb = self.forb_mask[active].astype(np.float64)
        shifted = sub_logits - np.max(sub_logits)
        p_base = np.exp(shifted)
        p_base = p_base / np.maximum(p_base.sum(), 1e-30)
        psi = np.sqrt(p_base + 1e-30).astype(np.complex128)
        log_p = np.log(p_base + 1e-30).astype(np.float64)
        V = -log_p - self.cfg.alpha_required * sub_req + self.cfg.beta_forbidden * sub_forb
        n_sub = sub_logits.shape[0]
        k_grid = np.fft.fftfreq(n_sub) * (2.0 * np.pi)
        psi = _evolve(psi, V, self.cfg, k_grid)
        prob = np.abs(psi) ** 2
        total = prob.sum()
        if not np.isfinite(total) or total <= 0:
            return scores
        prob = prob / total
        sub_new_logits = np.log(prob + 1e-30)
        new_logits_np = np_scores.copy()
        new_logits_np[active] = sub_new_logits
        out = torch.from_numpy(new_logits_np).to(scores.dtype).to(scores.device)
        return out.unsqueeze(0)

print('schrodinger primitive loaded')

In [ ]:
# Cell 7: Generator factories — load model, decode prompts, return raw responses
import gc, time
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList

MAX_NEW_TOKENS = 320

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def encode_prompt(tok, text):
    try:
        enc = tok.apply_chat_template(
            [{'role': 'user', 'content': text}],
            return_tensors='pt', add_generation_prompt=True,
        )
    except Exception:
        enc = tok(text, return_tensors='pt').input_ids
    if hasattr(enc, 'input_ids') and not hasattr(enc, 'shape'):
        enc = enc.input_ids
    return enc

def materialize_vocab(tok):
    n = tok.vocab_size
    out = []
    for i in range(n):
        try:
            s = tok.decode([i], skip_special_tokens=False)
        except Exception:
            s = ''
        out.append(s or '')
    return out

def run_ar(model_id, prompts):
    print(f'[ar] loading {model_id}')
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, trust_remote_code=True).eval().to('cuda')
    out = []
    for i, p in enumerate(prompts):
        enc = encode_prompt(tok, (p.get('prompt') or '').strip()).to('cuda')
        t0 = time.time()
        with torch.inference_mode():
            gen = model.generate(enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
        elapsed = time.time() - t0
        new_tokens = gen[0, enc.shape[1]:]
        response = tok.decode(new_tokens, skip_special_tokens=True)
        print(f"[ar] {i+1}/{len(prompts)} {p['id']} {elapsed:.1f}s ({len(response)} chars)")
        out.append({'label': 'ar', 'id': p['id'], 'response': response, 'seconds': elapsed})
    del model, tok
    free_memory()
    return out

def run_schrodinger(model_id, prompts, cfg=None):
    cfg = cfg or SchrodingerConfig()
    print(f'[schrodinger] loading {model_id}')
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, trust_remote_code=True).eval().to('cuda')
    print('[schrodinger] materializing vocab strings (one-time, ~30s)')
    vocab_strings = materialize_vocab(tok)
    out = []
    for i, p in enumerate(prompts):
        req_mask, forb_mask = _build_indicator_masks(vocab_strings, list(p.get('required') or []), list(p.get('forbidden') or []))
        processor = SchrodingerLogitsProcessor(req_mask, forb_mask, cfg)
        enc = encode_prompt(tok, (p.get('prompt') or '').strip()).to('cuda')
        t0 = time.time()
        with torch.inference_mode():
            gen = model.generate(
                enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tok.pad_token_id,
                logits_processor=LogitsProcessorList([processor]),
            )
        elapsed = time.time() - t0
        new_tokens = gen[0, enc.shape[1]:]
        response = tok.decode(new_tokens, skip_special_tokens=True)
        print(f"[schrodinger] {i+1}/{len(prompts)} {p['id']} {elapsed:.1f}s ({len(response)} chars)")
        out.append({'label': 'schrodinger', 'id': p['id'], 'response': response, 'seconds': elapsed})
    del model, tok
    free_memory()
    return out

def run_diffusion(model_id, prompts, num_steps=64):
    print(f'[diffusion] loading {model_id}')
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    try:
        model = AutoModel.from_pretrained(model_id, torch_dtype=torch.float16, trust_remote_code=True)
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, trust_remote_code=True)
    model = model.eval().to('cuda')
    out = []
    for i, p in enumerate(prompts):
        enc = encode_prompt(tok, (p.get('prompt') or '').strip()).to('cuda')
        t0 = time.time()
        kwargs = {'max_new_tokens': MAX_NEW_TOKENS, 'pad_token_id': tok.pad_token_id}
        try_kwargs = dict(kwargs, num_inference_steps=num_steps, do_sample=False)
        with torch.inference_mode():
            try:
                gen = model.generate(enc, **try_kwargs)
            except TypeError:
                gen = model.generate(enc, **kwargs)
        elapsed = time.time() - t0
        new_tokens = gen[0, enc.shape[1]:] if gen.dim() > 1 else gen
        response = tok.decode(new_tokens, skip_special_tokens=True)
        print(f"[diffusion] {i+1}/{len(prompts)} {p['id']} {elapsed:.1f}s ({len(response)} chars)")
        out.append({'label': 'diffusion', 'id': p['id'], 'response': response, 'seconds': elapsed})
    del model, tok
    free_memory()
    return out

print('generator factories ready')

In [ ]:
# Cell 8: Run AR baseline + Schrödinger oracle (both use Qwen-Coder-7B)
QWEN_7B = 'Qwen/Qwen2.5-Coder-7B-Instruct'

ar_raws = run_ar(QWEN_7B, CONTRACT['prompts'])
ar_verdicts = [gate_score(p, r['response']) for p, r in zip(CONTRACT['prompts'], ar_raws)]
ar_n_pass = sum(1 for v in ar_verdicts if v['ok'])
print(f'\n[ar] DONE: {ar_n_pass}/{len(CONTRACT["prompts"])}\n')

sw_raws = run_schrodinger(QWEN_7B, CONTRACT['prompts'])
sw_verdicts = [gate_score(p, r['response']) for p, r in zip(CONTRACT['prompts'], sw_raws)]
sw_n_pass = sum(1 for v in sw_verdicts if v['ok'])
print(f'\n[schrodinger] DONE: {sw_n_pass}/{len(CONTRACT["prompts"])}\n')

In [ ]:
# Cell 9: (OPTIONAL) Try DiffuCoder. Skip cell if it fails — bake-off still works with AR + Schrödinger.
DIFFUCODER = 'apple/DiffuCoder-7B-Instruct'
diff_raws = []
diff_verdicts = []
diff_n_pass = None
try:
    diff_raws = run_diffusion(DIFFUCODER, CONTRACT['prompts'])
    diff_verdicts = [gate_score(p, r['response']) for p, r in zip(CONTRACT['prompts'], diff_raws)]
    diff_n_pass = sum(1 for v in diff_verdicts if v['ok'])
    print(f'\n[diffusion] DONE: {diff_n_pass}/{len(CONTRACT["prompts"])}\n')
except Exception as exc:
    print(f'[diffusion] SKIPPED: {exc}')

In [ ]:
# Cell 10: Triangulate by shape + write JSON + Markdown report
import json
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

generators = [
    {'label': 'ar', 'model_id': QWEN_7B, 'n_pass': ar_n_pass, 'n_total': len(CONTRACT['prompts']), 'pass_rate': ar_n_pass/len(CONTRACT['prompts']), 'by_prompt': ar_verdicts},
    {'label': 'schrodinger', 'model_id': QWEN_7B, 'n_pass': sw_n_pass, 'n_total': len(CONTRACT['prompts']), 'pass_rate': sw_n_pass/len(CONTRACT['prompts']), 'by_prompt': sw_verdicts},
]
if diff_n_pass is not None:
    generators.append({'label': 'diffusion', 'model_id': DIFFUCODER, 'n_pass': diff_n_pass, 'n_total': len(CONTRACT['prompts']), 'pass_rate': diff_n_pass/len(CONTRACT['prompts']), 'by_prompt': diff_verdicts})

labels = [g['label'] for g in generators]
indexed = {g['label']: {v['id']: v for v in g['by_prompt']} for g in generators}
prompt_ids = [v['id'] for v in generators[0]['by_prompt']]
by_shape = {}
by_prompt_tri = []
for pid in prompt_ids:
    verdicts = {lab: indexed[lab].get(pid) for lab in labels}
    shape = next((v['shape'] for v in verdicts.values() if v), 'unknown')
    bucket = by_shape.setdefault(shape, {lab: {'pass': 0, 'fail': 0} for lab in labels})
    for lab in labels:
        v = verdicts[lab]
        bucket[lab]['pass' if (v and v['ok']) else 'fail'] += 1
    winners = [lab for lab in labels if verdicts[lab] and verdicts[lab]['ok']]
    losers = [lab for lab in labels if verdicts[lab] and not verdicts[lab]['ok']]
    if not winners: cls = 'all_fail'
    elif not losers: cls = 'all_pass'
    else: cls = 'split'
    by_prompt_tri.append({'id': pid, 'shape': shape, 'verdict_class': cls, 'winners': winners, 'losers': losers})

shape_delta = {}
if 'ar' in labels and 'schrodinger' in labels:
    for shape, counts in by_shape.items():
        shape_delta[shape] = {'ar_pass': counts['ar']['pass'], 'schrodinger_pass': counts['schrodinger']['pass'], 'delta_sw_minus_ar': counts['schrodinger']['pass'] - counts['ar']['pass']}

report = {
    'schema_version': 'scbe_diffusion_bakeoff_report_v1',
    'contract_id': CONTRACT['contract_id'],
    'source_contract': CONTRACT['source_contract'],
    'stamp': stamp,
    'n_prompts': len(CONTRACT['prompts']),
    'generators': generators,
    'triangulation': {'by_shape': by_shape, 'by_prompt': by_prompt_tri, 'shape_delta': shape_delta},
}

json_path = f'/content/diffusion_bakeoff_{stamp}.json'
raws_path = f'/content/diffusion_bakeoff_{stamp}.responses.jsonl'
md_path = f'/content/diffusion_bakeoff_{stamp}.md'
with open(json_path, 'w') as f:
    json.dump(report, f, indent=2)
with open(raws_path, 'w') as f:
    for raws in (ar_raws, sw_raws, diff_raws):
        for row in raws:
            f.write(json.dumps(row) + '\n')

lines = [f'# Code-Diffusion Bake-Off — {report["contract_id"]}', '', '## Generator pass-rates', '', '| Generator | Model | Pass | Rate |', '|---|---|---|---|']
for g in generators:
    lines.append(f'| {g["label"]} | `{g["model_id"]}` | {g["n_pass"]}/{g["n_total"]} | {g["pass_rate"]:.3f} |')
if shape_delta:
    lines += ['', '## Per-shape delta (Schrödinger - AR)', '', '| Shape | AR pass | Schrödinger pass | Delta |', '|---|---|---|---|']
    for shape, row in sorted(shape_delta.items(), key=lambda kv: -kv[1]['delta_sw_minus_ar']):
        lines.append(f'| {shape} | {row["ar_pass"]} | {row["schrodinger_pass"]} | {row["delta_sw_minus_ar"]:+d} |')
lines += ['', '## Per-prompt verdict-class', '', '| Prompt | Shape | Class | Winners |', '|---|---|---|---|']
for row in by_prompt_tri:
    winners = ', '.join(row['winners']) or '—'
    lines.append(f'| {row["id"]} | {row["shape"]} | {row["verdict_class"]} | {winners} |')
with open(md_path, 'w') as f:
    f.write('\n'.join(lines) + '\n')

print('wrote:', json_path)
print('wrote:', raws_path)
print('wrote:', md_path)
print()
print(open(md_path).read())

In [ ]:
# Cell 11: Push results to private HF dataset
from huggingface_hub import HfApi, create_repo

RESULTS_REPO = 'issdandavis/scbe-diffusion-bakeoff-results'
api = HfApi(token=HF_TOKEN)
create_repo(RESULTS_REPO, repo_type='dataset', private=True, exist_ok=True, token=HF_TOKEN)
for fp in [json_path, raws_path, md_path]:
    name = fp.split('/')[-1]
    target = f'runs/{stamp}/{name}'
    api.upload_file(path_or_fileobj=fp, path_in_repo=target, repo_id=RESULTS_REPO, repo_type='dataset', token=HF_TOKEN)
    print(f'uploaded {target}')
print(f'\nResults at: https://huggingface.co/datasets/{RESULTS_REPO}/tree/main/runs/{stamp}')